In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import scanpy as sc
import anndata

from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/higashi/csv/clusters/leiden_with_ips_main_alt.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.processed.harmony.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.head()

In [ ]:
adata.obs.index = [x[:-2] for x in adata.obs.index]

In [ ]:
adata.obs['renamed_leiden'] = clusters.reindex(adata.obs.index)['renamed_leiden']
adata.obs['ips_cluster_time'] = clusters.reindex(adata.obs.index)['ips_cluster_time']

In [ ]:
sc.pl.umap(adata, color=['assay', 'time', 'line', 'renamed_leiden'], ncols=2)

### predict

In [ ]:
snm3c = adata[adata.obs['assay'] == 'snm3C']
snm3c = snm3c[~snm3c.obs['renamed_leiden'].isna()]
snm3c

In [ ]:
snm3c.obs['train_leiden'] = snm3c.obs['renamed_leiden'].astype(str)
mask = [x.split('_')[0] != 'IPS' for x in snm3c.obs['train_leiden']]
snm3c.obs.loc[mask,'train_leiden'] = 'other'

In [ ]:
snm3c.obs['train_leiden'].value_counts()

In [ ]:
### all labels
features = np.hstack([snm3c.obsm['X_pca_harmony']])

X = features
y = snm3c.obs['renamed_leiden']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)
xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

In [ ]:
### no time

features = np.hstack([snm3c.obsm['X_pca_harmony']])

X = features
y = snm3c.obs['train_leiden']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)
xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

In [ ]:
features = np.hstack([snm3c.obsm['X_pca_harmony'], snm3c.obs['time'].values.reshape(-1, 1)])

X = features
y = snm3c.obs['train_leiden']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)
xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

### try the time corrected?

In [ ]:
### all labels
features = np.hstack([snm3c.obsm['X_pca_harmony']])

X = features
y = snm3c.obs['ips_cluster_time']
train_size = 0.8
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size)
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X_train, y_train)
xgboost_pred = model.predict(X_test)
print(classification_report(y_test, xgboost_pred))

### apply model to entire snm3c and transfer

In [ ]:
snmct = adata[adata.obs['assay'] == 'snmCT']
snmct.shape

In [ ]:
features = np.hstack([snm3c.obsm['X_pca_harmony']])
X = features
y = snm3c.obs['ips_cluster_time']
model = OneVsRestClassifier(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
model.fit(X, y)

In [ ]:
X_snmct = snmct.obsm['X_pca_harmony']

snmct_labels = model.predict(X_snmct)
snmct_labels.shape

In [ ]:
label_df = pd.DataFrame(index=snmct.obs.index)
label_df['snmct_pred'] = snmct_labels
label_df['snmct_pred'].value_counts()

In [ ]:
adata.obs['snmct_pred'] = label_df.reindex(adata.obs.index)['snmct_pred']

In [ ]:
sc.pl.umap(adata, color=['renamed_leiden', 'snmct_pred'], ncols=2)

In [ ]:
mask1 = [str(x).split('_')[0] == 'IPS' for x in adata.obs['snmct_pred']]
mask2 = [str(x).split('_')[0] == 'IPS' for x in adata.obs['renamed_leiden']]

mask = np.logical_or(mask1, mask2)
ips_adata = adata[mask]
ips_adata

In [ ]:
sc.pl.umap(ips_adata, color=['renamed_leiden', 'snmct_pred'], ncols=2)

In [ ]:
sc.pl.umap(ips_adata, color=['renamed_leiden', 'snmct_pred'], groups = 'IPS_alt', ncols=2)

In [ ]:
sc.pl.umap(ips_adata, color=['renamed_leiden', 'snmct_pred'], groups = 'IPS_main', ncols=2)

In [ ]:
tmp_meta = ips_adata.obs.copy()
tmp_meta['ips_subtype'] = tmp_meta['snmct_pred'].astype(str)
mask = ~tmp_meta['renamed_leiden'].isna()
tmp_meta.loc[mask,'ips_subtype'] = tmp_meta.loc[mask,'renamed_leiden']

###
mask = tmp_meta['ips_cluster_time'].isna()
tmp_meta.loc[mask,'ips_cluster_time'] = tmp_meta.loc[mask,'ips_subtype']
tmp_meta.shape

In [ ]:
tmp_meta['ips_subtype'].value_counts(), \
tmp_meta['ips_cluster_time'].value_counts()

In [ ]:
tmp_meta.to_csv(f'{projdir}/csv/label_transfer/snm3c_ips_subtype.csv', sep='\t')

In [ ]:
!date